# Milestone 5 lab, notebook 1 of 8: Did it run?

## Where we are

| Notebook | What you do |
|---|---|
| **1. Did it run?** (you are here) | Read Airflow's run history and the pipeline's own. Run one night twice |
| 2. On time, as a number | Measure freshness. Turn "on time" into a number with a target |
| 3. Five signals | Monitoring asks if the job ran. Observability asks if the data is right |
| 4. Tell the person who can act | Airflow calls back, and the right person gets an alert |
| 5. Too many alerts | Count the careless way. Then group, send once, rank and silence |
| 6. Fix one day | Fix the source, run one night again, prove the number |
| 7. Four drills | Four faults on a clean day, each caught and repaired. Write an incident note |
| 8. The Monday answer | The whole week, the recovery, and the answer to the question |

**You need:** the warehouse and Airflow running. From the project folder: `docker compose up -d --build`. Airflow's page is http://localhost:8080 .

**About ten minutes of reading, and two minutes of waiting** while Airflow runs four nights. Run the cells one by one.

In [1]:
import pandas as pd
from lab_helpers import (
    MON, TUE, WED, airflow_runs, airflow_tasks, airflow_tools, health_numbers, run_nights, sql,
    start_again, store_day,
)

airflow_tools.wait_until_ready()

## Step 1: Where milestone 4 stopped

**Lotus Retails** is a chain of six supermarkets, `S01` to `S06`. Customers pay at a **till**: the machine at the counter where a customer pays. It is also called a cash register.

**Ines Marchetti** runs store `S04`. On Tuesday 25 August her tills took **$587.42**. The file with her sales lines never reached the pipeline.

The pipeline of milestone 4 saw it that same night, at 02:00. A gate wrote one correct row into a table: "`S04`, Tuesday, no sales file". Then nobody read that row for six days. On Monday the bonus report said "do not pay yet", and that was the first time anyone heard of it.

The report was right. Nobody was told. This last milestone fixes that, in three parts:

| Part | The question | Notebooks |
|---|---|---|
| **Know** | Is the pipeline healthy? Did it run, on time, with every store? | 1, 2, 3 |
| **Alert** | Who must be told, how urgently, and what should they do? | 4, 5 |
| **Recover** | A store sends a corrected file. How does the right number come back? | 6, 7, 8 |

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, what happens when it fails, and who is told.

This notebook asks the first question of all: **did it run?**

## Step 2: Start from nothing, and run Monday

**Airflow** is a scheduler: a program that starts other programs at the right time and in the right order. Our pipeline is one Airflow **DAG**: a list of tasks, and which task waits for which. It has eight tasks. The first one, `stores_send_files`, plays the six stores sending their files. The other seven are the pipeline you know: load, a gate, clean, a gate, build, a gate, publish.

In real life Airflow starts the DAG at 02:00 every night. Here you do not wait for 02:00: `run_night` asks Airflow to run one night now, and waits until it ends.

First, empty everything this milestone has: the warehouse, the stores' files, and what Airflow remembers. Then run Monday 24 August.

In [2]:
start_again()
pd.DataFrame(airflow_tools.run_night("2026-08-24"))

Empty: the warehouse of this milestone, the stores' landing folder, the alerts log,
and Airflow's runs of lotus_nightly_m5.


,task,state
0,stores_send_files,success
1,load,success
2,gate_landing,success
3,clean,success
4,gate_clean,success
5,build,success
6,gate_publish,success
7,publish,success


Eight tasks, eight times `success`.

## Step 3: Two more nights

Run Tuesday and Wednesday. After each night, the last column asks the bonus question: how many of the six stores could be paid right now?

In [3]:
run_nights([TUE, WED])

,night,tasks,succeeded,seconds,stores that can be paid
0,Tue 08-25,8,8,23,4
1,Wed 08-26,8,8,25,4


Every task green, both nights. And yet only four stores can be paid. Here is Ines's Tuesday, as the pipeline published it:

In [4]:
store_day("S04", TUE)

,store_id,sales_date,lines_cents,till_cents,rows_set_aside,status,problem
0,S04,2026-08-25,0,58742,0,do not pay yet,no_sales_file


`lines_cents` is 0: no sales lines. `till_cents` is 58742: the tills counted $587.42. The status says `do not pay yet`, because `no_sales_file`.

So "did it run?" has a simple answer, yes, and it is not enough. Keep that in mind for notebooks 2 and 3. First, learn where the answer "yes" comes from.

## Step 4: Airflow's run history

Airflow remembers every run of a DAG: which night it was for, how it ended, and when it ran. This is its **run history**.

In [5]:
airflow_runs()

,run_id,for the night of,state,started (UTC),seconds
0,night_2026-08-24,2026-08-24,success,2026-11-25 15:29:50,12
1,night_2026-08-25,2026-08-25,success,2026-11-25 15:30:13,11
2,night_2026-08-26,2026-08-26,success,2026-11-25 15:30:37,12


- `run_id` is the run's name. `run_night` names each run `night_` and the day.
- `for the night of` is the business day: the day the stores were open. Airflow calls it the *logical date*. The run itself happens at 02:00 the morning after.
- `started (UTC)` is the real clock of this computer, in UTC, the world's standard time. It is today, not August 2026.
- `seconds` is how long the whole run took.

Airflow also remembers each task of each run:

In [6]:
airflow_tasks(TUE)

,task,state,started (UTC),seconds
0,stores_send_files,success,15:30:13,0.6
1,load,success,15:30:14,0.2
2,gate_landing,success,15:30:15,1.4
3,clean,success,15:30:17,0.2
4,gate_clean,success,15:30:18,2.2
5,build,success,15:30:20,0.2
6,gate_publish,success,15:30:21,1.5
7,publish,success,15:30:23,0.3


**Look at it yourself:** open http://localhost:8080 , click **Dags**, then `lotus_nightly_m5`. Each night is a column of green boxes. Click a box to read that task's log.

## Step 5: The pipeline's own run history

The pipeline keeps a run history of its own, in the warehouse: the table `ops.run_history`. Every step of every run leaves one row, and no row is ever deleted.

In [7]:
sql("""
    SELECT run_day, step, status, started_at, seconds, detail
    FROM ops.run_history
    WHERE run_day = ?
    ORDER BY started_at
""", (TUE,))

,run_day,step,status,started_at,seconds,detail
0,2026-08-25,load,ok,2026-11-25 15:30:14.635767,0.08,"load: 21 new files, 1,099 rows copied to the raw tables."
1,2026-08-25,gate-landing,ok,2026-11-25 15:30:15.674999,1.19,"gate landing: 4 tests, 1 passed."
2,2026-08-25,clean,ok,2026-11-25 15:30:17.608158,0.11,"clean: 7 clean tables rebuilt. sales_lines has 20,973 rows."
3,2026-08-25,gate-clean,ok,2026-11-25 15:30:18.655712,1.95,"gate clean: 40 tests, 37 passed."
4,2026-08-25,build,ok,2026-11-25 15:30:20.895371,0.08,build: staging.daily_store_sales has 179 rows. Nobody reads staging.
5,2026-08-25,gate-publish,ok,2026-11-25 15:30:21.754369,1.23,"gate publish: 5 tests, 4 passed."
6,2026-08-25,publish,ok,2026-11-25 15:30:23.810594,0.06,publish: daily_store_sales has 179 rows. replenishment_list has 234 rows for tomorrow. 2 store days are not rea...
7,2026-08-25,report,ok,2026-11-25 15:30:23.927508,0.06,"Lotus Retails. Store managers' bonus, week of Mon 2026-08-24"


`detail` is the first line the step printed. The load step says how many files and rows it copied. A gate says how many tests passed.

Why keep two histories? Each one sees something the other does not.

| | Airflow's run history | The pipeline's `ops.run_history` |
|---|---|---|
| Where | Airflow's own database | The warehouse, beside the data |
| One row per | Run, and task of a run | Step of a run |
| Sees `stores_send_files` | Yes. It is an Airflow task | No. It is the stores, not the pipeline |
| Sees `report` | No. It runs inside the `publish` task | Yes. It is a step of its own |
| A step run by hand, without Airflow | Not seen | Seen: `uv run python -m lotus_pipeline load --day 2026-08-25` leaves a row too |
| What the step said | In the task's log | In `detail`, one line, easy to query |

## Step 6: Run one night twice

Somebody wonders: "Maybe Ines's file came in by now. Run last night again." Every run first collects every file that has arrived, so a late Tuesday file would be picked up. Run Wednesday, the newest night, a second time, and watch both histories.

In [8]:
before = airflow_runs()
pd.DataFrame(airflow_tools.run_night("2026-08-26"))

,task,state
0,stores_send_files,success
1,load,success
2,gate_landing,success
3,clean,success
4,gate_clean,success
5,build,success
6,gate_publish,success
7,publish,success


In [9]:
after = airflow_runs()
wednesday = "2026-08-26"
pd.concat([
    before[before["for the night of"] == wednesday].assign(seen="before the second run"),
    after[after["for the night of"] == wednesday].assign(seen="after the second run"),
], ignore_index=True)

,run_id,for the night of,state,started (UTC),seconds,seen
0,night_2026-08-26,2026-08-26,success,2026-11-25 15:30:37,12,before the second run
1,night_2026-08-26,2026-08-26,success,2026-11-25 15:31:09,12,after the second run


Airflow still has **one** run for Wednesday, with the same `run_id`. Its start time is now the second run's. The first run's times are gone from this list. (Airflow does keep the log of each try of each task, one file per try.)

Now the pipeline's own history:

In [10]:
sql("""
    SELECT run_day, step, status, started_at, seconds, detail
    FROM ops.run_history
    WHERE run_day = ? AND step IN ('load', 'publish')
    ORDER BY started_at
""", (WED,))

,run_day,step,status,started_at,seconds,detail
0,2026-08-26,load,ok,2026-11-25 15:30:38.721681,0.11,"load: 21 new files, 830 rows copied to the raw tables."
1,2026-08-26,publish,ok,2026-11-25 15:30:48.712917,0.07,publish: daily_store_sales has 185 rows. replenishment_list has 234 rows for tomorrow. 2 store days are not rea...
2,2026-08-26,load,ok,2026-11-25 15:31:10.417509,0.07,"load: 0 new files, 0 rows copied to the raw tables."
3,2026-08-26,publish,ok,2026-11-25 15:31:19.772732,0.07,publish: daily_store_sales has 185 rows. replenishment_list has 234 rows for tomorrow. 2 store days are not rea...


Both runs are there. Read the two `load` rows. The first one copied the files of Wednesday night. The second one says **0 new files**: nothing had changed, so nothing was loaded twice. The two `publish` rows say the same. Remember that. Notebook 6 is built on it.

The pipeline can also answer for each night: did its newest run publish, how long did it take, and how many runs did it have?

In [11]:
pd.DataFrame(health_numbers(WED)["night_runs"])

,run_day,published,seconds,runs,blocked_at
0,2026-08-24,True,5.2,1,None
1,2026-08-25,True,4.8,1,None
2,2026-08-26,True,4.9,2,None


`runs` is 2 for Wednesday, 1 for the others. And look at `S04` once more: running again did not make her Tuesday file appear.

In [12]:
store_day("S04", TUE)[["store_id", "sales_date", "status", "problem"]]

,store_id,sales_date,status,problem
0,S04,2026-08-25,do not pay yet,no_sales_file


A second run is safe. It is not a cure. The file has to come from the store, and somebody has to ask for it. That is the **alert** part of this milestone.

## What you now know

- **Run history** is what a pipeline remembers about its own runs: which night, which step, how it ended, how long it took.
- Airflow keeps one: one run per night, and one row per task. A night run again keeps its one row, with the newest times.
- The pipeline keeps its own in `ops.run_history`: one row per step of every run, never deleted. It also sees steps run by hand.
- Every task was green, and a store still could not be paid. "Did it run?" is the first question, not the last.

**Say it in an interview:** "Every step of every run left a row, and nothing was deleted. When a night was run twice I could see both runs, and that the second load found nothing new. Airflow kept one row per night with the newest times, so I kept our own history beside the data too."

**The theory:** [Data Observability & Monitoring](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-observability-and-monitoring), and [Airflow: Monitoring, Logging, Alerting & Production Failures](https://www.d8loop.com/modules/knowledge-corner/airflow/monitoring-logging-alerting-and-production-failures).

Next: [notebook 2](02_on_time_as_a_number.ipynb). Every task was green. Was every file on time?